In [0]:
import zipfile
import os
import requests

catalog = "workspace"
schema = "ans_data"
volume = "ans_extract"

url_ans = "https://dadosabertos.ans.gov.br/FTP/PDA/informacoes_consolidadas_de_beneficiarios-024/202508/pda-024-icb-TO-2025_08.zip"

volume_path = f"/Volumes/{catalog}/{schema}/{volume}"
zip_file_name = "pda-024-icb-RS-2025_08.zip"
zip_full_path = os.path.join(volume_path, zip_file_name)
output_folder = os.path.join(volume_path, "raw_data")

os.makedirs(output_folder, exist_ok=True)

# Download file
def download_arquivo(url, destino):
    try:
        print(f"Iniciando download de: {url}")
        with requests.get(url, stream=True, timeout=30) as r:
            r.raise_for_status()
            with open(destino, 'wb') as f:
                for chunk in r.iter_content(chunk_size=1024 * 1024):
                    if chunk:
                        f.write(chunk)
        print("Downloaded!")
    except Exception as e:
        print(f"Download error: {e}")
        if not os.path.exists(destino):
            raise Exception("File not found.")

if not os.path.exists(zip_full_path):
    download_arquivo(url_ans, zip_full_path)
else:
    print(f"File exists {zip_full_path}.")


print(f" {zip_file_name}")
with zipfile.ZipFile(zip_full_path, 'r') as z:
    for member in z.infolist():
        if member.filename.endswith('.csv'):
            target_path = os.path.join(output_folder, member.filename)
            print(f"{member.filename}...")
            
            with z.open(member) as source, open(target_path, "wb") as target:
                while True:
                    chunk = source.read(1024 * 1024)
                    if not chunk:
                        break
                    target.write(chunk)

print("✅ Extract success!")

# Raed Spark
df = (spark.read
      .format("csv")
      .option("header", "true")
      .option("sep", ";")
      .option("inferSchema", "true")
      .load(output_folder))

# Write Bronze
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.bronze")
table_full_name = f"{catalog}.bronze.beneficiarios_ans"

df.write.format("delta").mode("overwrite").saveAsTable(table_full_name)

print(f"✅ Tabela {table_full_name} criada com {df.count()} linhas.")
display(spark.table(table_full_name).limit(5))